# RetroSpec v3 - Notebook 2: full comparative benchmark

Runs every method **in one process, against one model load, interleaved, with
warmup and repeated timing**, over your 500-question dataset.

Do not run this until Notebook 1's correctness gate passes.


In [ ]:
# --- dependencies -------------------------------------------------------
!pip install -q -U "transformers>=4.45" accelerate bitsandbytes rouge-score 2>&1 | tail -1
# faiss is optional; the hybrid drafter falls back to numpy, which is faster
# than faiss at these index sizes anyway.


In [ ]:
import os, sys, json, glob

# Point this at wherever the `retrospec` package lives.
CANDIDATES = ["/kaggle/working/RetroSpecV2", "/kaggle/input/retrospec", ".", ".."]
REPO = next((p for p in CANDIDATES if os.path.isdir(os.path.join(p, "retrospec"))), None)
if REPO is None:
    REPO = "/kaggle/working/RetroSpecV2"
    os.system(f"git clone -q https://github.com/lxzy8/RetroSpecV2.git {REPO}")
sys.path.insert(0, REPO)
os.chdir(REPO)

from retrospec import bench, data
from retrospec.engine import TorchVerifier, speculative_generate, greedy_generate
from retrospec.drafters import NGramDrafter, HybridDrafter, ModelDrafter, EarlyExitDrafter
from retrospec.calm import calm_generate

print("repo:", REPO)
print(bench.describe_gpu())


In [ ]:
# Llama-3.2 is gated. Store your token as a Kaggle Secret named HF_TOKEN
# (Add-ons -> Secrets), or paste it below.
try:
    from kaggle_secrets import UserSecretsClient
    tok_str = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login; login(tok_str)
    print("logged in to Hugging Face")
except Exception as e:
    print("No HF_TOKEN secret found:", e)
    print("Either add the secret, or switch VERIFIER/DRAFTER below to an "
          "ungated pair such as Qwen/Qwen2.5-1.5B-Instruct + Qwen/Qwen2.5-0.5B-Instruct.")


In [ ]:
VERIFIER_NAME = "meta-llama/Llama-3.2-3B-Instruct"
DRAFTER_NAME  = "meta-llama/Llama-3.2-1B-Instruct"   # same tokenizer family -- required
DEVICE        = "cuda:0"                             # pin to ONE gpu, never device_map="auto"
MAX_NEW       = 128                                  # 30 was far too short to measure anything

import torch
DTYPE = bench.pick_dtype()
print("dtype:", DTYPE, "| bf16 hardware:", bench.describe_gpu().get("bf16"))

verifier_model, tok = bench.load_model(VERIFIER_NAME, device=DEVICE, dtype=DTYPE)
V   = TorchVerifier(verifier_model, device=DEVICE)
EOS = bench.eos_ids_for(tok, verifier_model)
SYNC = bench.sync_fn()
print("eos ids:", EOS, "| layers:", len(verifier_model.model.layers))


## Dataset

Point `CSV` at your file. The loader auto-detects the question column and, if
the CSV has a context/passage column, marks those rows as *grounded*.

In [ ]:
CSV = "/kaggle/input/datasets/totaldose/ques-500/dataset_500.csv"
N_PROMPTS = 60          # 60 x ~6 methods x 3 repeats is a few GPU-hours; raise once it is green

print(os.popen("ls -R /kaggle/input 2>/dev/null | head -30").read())

prompts = data.load_prompts(CSV, n=N_PROMPTS, seed=0)

# Retrieval-based drafting only wins when the output reuses spans from the input.
# Open Q&A gives n-gram and hybrid nothing to retrieve. Add a grounded slice so
# the comparison can actually distinguish the methods.
GROUNDED = True
if GROUNDED:
    prompts = prompts + data.make_grounded_variants(prompts[:N_PROMPTS//2])
print(f"{len(prompts)} prompts ({sum(p['grounded'] for p in prompts)} grounded)")


## Load the calibrated gammas

In [ ]:
try:
    cfg = json.load(open("configs/calibration_v3.json"))
    GAMMA = cfg["gamma"]
except Exception:
    GAMMA = {"ngram": 6, "hybrid": 5, "spec_1b": 5}
    print("No calibration file -- using defaults. Run Notebook 1 first.")
print("gamma:", GAMMA)

REPEATS = 3


## Method definitions

In [ ]:
all_rows = []

# ---- 0. baseline: greedy, through the SAME engine as everything else -------
def run_baseline(ids):
    return greedy_generate(V, ids, max_new_tokens=MAX_NEW, eos_ids=EOS, sync=SYNC)

print("baseline")
base_rows = bench.run_method("baseline", run_baseline, prompts, tok,
                             max_new_tokens=MAX_NEW, repeats=REPEATS, verbose=False)
bench.attach_quality(base_rows, base_rows)
all_rows += base_rows
print(f"  mean {sum(r['tokens_per_sec'] for r in base_rows)/len(base_rows):.1f} tok/s")


In [ ]:
# ---- 1. n-gram (exact suffix lookup) --------------------------------------
_ng = {}
def run_ngram(ids):
    _ng["d"] = NGramDrafter(draft_len=GAMMA["ngram"])
    return speculative_generate(V, ids, draft_fn=_ng["d"], max_new_tokens=MAX_NEW,
                                eos_ids=EOS, sync=SYNC)

print("ngram")
r = bench.run_method("ngram", run_ngram, prompts, tok, repeats=REPEATS, verbose=False)
all_rows += bench.attach_quality(r, base_rows)


In [ ]:
# ---- 2. hybrid (n-gram + dense hidden-state kNN, RRF) ---------------------
_hy = {}
def run_hybrid(ids):
    d = HybridDrafter(draft_len=GAMMA["hybrid"]); _hy["d"] = d
    return speculative_generate(V, ids, draft_fn=d, max_new_tokens=MAX_NEW, eos_ids=EOS,
                                on_accept=lambda t, h: d.add(t, h),
                                need_hidden=True, sync=SYNC)

print("hybrid")
r = bench.run_method("hybrid", run_hybrid, prompts, tok, repeats=REPEATS, verbose=False)
all_rows += bench.attach_quality(r, base_rows)


In [ ]:
# ---- 3. spec_1b: Llama-3.2-1B drafter, bf16/fp16 --------------------------
drafter_model, _ = bench.load_model(DRAFTER_NAME, device=DEVICE, dtype=DTYPE)
_md = ModelDrafter(drafter_model, draft_len=GAMMA["spec_1b"], device=DEVICE)

def run_spec1b(ids):
    _md.reset()
    return speculative_generate(V, ids, draft_fn=_md, max_new_tokens=MAX_NEW,
                                eos_ids=EOS, sync=SYNC)

print("spec_1b")
r = bench.run_method("spec_1b", run_spec1b, prompts, tok, repeats=REPEATS,
                     reset=_md.reset, verbose=False)
all_rows += bench.attach_quality(r, base_rows)


In [ ]:
# ---- 4. spec_1b_4bit: quantised drafter (memory-lean variant) -------------
del drafter_model, _md; bench.free()
q_model, _ = bench.load_model(DRAFTER_NAME, quantize_4bit=True, device=DEVICE, dtype=DTYPE)
_q = ModelDrafter(q_model, draft_len=GAMMA["spec_1b"], device=DEVICE)

def run_spec1b_q(ids):
    _q.reset()
    return speculative_generate(V, ids, draft_fn=_q, max_new_tokens=MAX_NEW,
                                eos_ids=EOS, sync=SYNC)

print("spec_1b_4bit")
r = bench.run_method("spec_1b_4bit", run_spec1b_q, prompts, tok, repeats=REPEATS,
                     reset=_q.reset, verbose=False)
all_rows += bench.attach_quality(r, base_rows)
del q_model, _q; bench.free()


In [ ]:
# ---- 5. early exit as a lossless drafter (supersedes v2's CALM) -----------
try:
    _ee = EarlyExitDrafter(verifier_model, n_layers=len(verifier_model.model.layers)//2,
                           draft_len=4, device=DEVICE)
    def run_ee(ids):
        _ee.reset()
        return speculative_generate(V, ids, draft_fn=_ee, max_new_tokens=MAX_NEW,
                                    eos_ids=EOS, sync=SYNC)
    print("earlyexit_spec")
    r = bench.run_method("earlyexit_spec", run_ee, prompts, tok, repeats=REPEATS,
                         reset=_ee.reset, verbose=False)
    all_rows += bench.attach_quality(r, base_rows)
except Exception as e:
    print("skipped earlyexit_spec:", type(e).__name__, e)


In [ ]:
# ---- 6. CALM (lossy). Reported for QUALITY drift, not speed ---------------
L = len(verifier_model.model.layers) // 2
def run_calm(ids):
    return calm_generate(verifier_model, ids, exit_layer=L, confidence_threshold=0.8,
                         max_new_tokens=MAX_NEW, eos_ids=EOS, device=DEVICE, sync=SYNC)

print("calm")
r = bench.run_method("calm", run_calm, prompts, tok, repeats=REPEATS,
                     lossless=False, verbose=False)
all_rows += bench.attach_quality(r, base_rows)


## Results

In [ ]:
df, agg = bench.summarize(all_rows)
print(agg.to_string())
bench.sanity_check(agg)
bench.save(all_rows, "results/v3_results.json")


In [ ]:
# Grounded vs open: this is the slice that separates retrieval drafting from
# model drafting. Averaging over both is what made v2's story unreadable.
import pandas as pd
slice_df = df.groupby(["method", "grounded"]).agg(
    speedup=("speedup", "mean"),
    tok_per_fwd=("tokens_per_forward", "mean"),
    acceptance=("acceptance_rate", "mean"),
).round(3).unstack()
print(slice_df.to_string())


In [ ]:
import matplotlib.pyplot as plt

order = agg.index.tolist()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
agg.loc[order, "speedup_mean"].plot.bar(ax=ax[0], color="#4C78A8")
ax[0].axhline(1.0, color="crimson", ls="--", lw=1); ax[0].set_title("Wall-clock speedup vs greedy")
agg.loc[order, "tok_per_fwd"].plot.bar(ax=ax[1], color="#72B7B2")
ax[1].axhline(1.0, color="crimson", ls="--", lw=1); ax[1].set_title("Tokens per verifier forward (ceiling)")
agg.loc[order, "exact_match"].plot.bar(ax=ax[2], color="#E45756")
ax[2].set_ylim(0, 1.05); ax[2].set_title("Exact match vs greedy (token ids)")
for a in ax: a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("results/v3_summary.png", dpi=140); plt.show()


## What the table should look like

| method | expected | if it does not |
|---|---|---|
| `spec_1b` | **1.6-2.4x**, exact_match 1.0 | drafter too slow, or gamma too large |
| `earlyexit_spec` | 1.2-1.7x, exact_match 1.0 | try fewer draft layers |
| `ngram` / `hybrid` | ~1.0x on open Q&A, **1.3-2.0x on the grounded slice** | that gap *is* the result -- report it |
| `spec_1b_4bit` | slower than `spec_1b`, lower VRAM | expected; nf4 dequantisation costs more than it saves at batch 1 |
| `calm` | < 1.0x, exact_match < 1.0 | expected and explained in `retrospec/calm.py` |

`exact_match` must be exactly 1.0 for every method marked lossless. Anything
else is a bug, not a result.
